# Вебинар 4: Машинное обучение без учителя

## 📊 Датасет 1: Mall Customers (сегментация)
**Название:** [Mall Customers](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python)  
**Описание:** 200 клиентов торгового центра с признаками: возраст, доход, гендер, spending score.  
**Задача:** Сегментация клиентов, поиск аномалий.

## 📊 Датасет 2: Credit Card (аномалии)
**Название:** [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Задача:** Обнаружение аномалий (unsupervised) — Isolation Forest, LOF.

## 🎯 Цели ноутбука
1. Сегментация клиентов через K-Means, DBSCAN
2. Бизнес-интерпретация кластеров
3. Обнаружение аномалий (Isolation Forest, LOF)
4. Снижение размерности (PCA, t-SNE)

## Часть 1: Сегментация клиентов

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

# Загрузка датасета Mall Customers
df = pd.read_csv('/kaggle/input/customer-segmentation-tutorial-in-python/Mall_Customers.csv')
print(f"Размер: {df.shape}")
df.head()

In [ ]:
print("Распределение по полу:")
print(df['Genre'].value_counts())
print("\nСтатистика:")
print(df.describe())

In [ ]:
# Подготовка данных: только числовые признаки
df_features = df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features)

# Визуализация
fig = plt.figure(figsize=(15, 5))

ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(df['Age'], df['Annual Income (k$)'], df['Spending Score (1-100)'], c='steelblue')
ax1.set_xlabel('Age'); ax1.set_ylabel('Income'); ax1.set_zlabel('Spending')
ax1.set_title('3D визуализация клиентов')

ax2 = fig.add_subplot(132)
ax2.scatter(df['Age'], df['Annual Income (k$)'], alpha=0.6)
ax2.set_xlabel('Age'); ax2.set_ylabel('Annual Income')
ax2.set_title('Age vs Income')

ax3 = fig.add_subplot(133)
ax3.scatter(df['Annual Income (k$)'], df['Spending Score (1-100)'], alpha=0.6)
ax3.set_xlabel('Annual Income'); ax3.set_ylabel('Spending Score')
ax3.set_title('Income vs Spending')

plt.tight_layout()
plt.show()

## 2. K-Means: выбор оптимального K

In [ ]:
# Метод локтя и silhouette
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'o-', linewidth=2, markersize=8)
axes[0].set_xlabel('K (число кластеров)')
axes[0].set_ylabel('Inertia (внутрикластерная сумма квадратов)')
axes[0].set_title('Метод локтя')
axes[0].grid(alpha=0.3)

axes[1].plot(K_range, silhouettes, 'o-', linewidth=2, markersize=8, color='orange')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')
axes[1].grid(alpha=0.3)

optimal_k = K_range[np.argmax(silhouettes)]
print(f"Оптимальное K: {optimal_k}")

plt.tight_layout()
plt.show()

In [ ]:
# Финальная кластеризация
km_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['cluster'] = km_final.fit_predict(X_scaled)

print("Профили кластеров:")
profile = df.groupby('cluster').agg({
    'Age': 'mean',
    'Annual Income (k$)': 'mean',
    'Spending Score (1-100)': 'mean',
    'Genre': lambda x: (x == 'Male').mean(),
    'CustomerID': 'count'
}).rename(columns={'CustomerID': 'count', 'Genre': 'pct_male'})
print(profile.round(2))

In [ ]:
# Визуализация кластеров
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

cluster_colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown']

for c in range(optimal_k):
    cluster_data = df[df['cluster'] == c]
    axes[0].scatter(cluster_data['Annual Income (k$)'], cluster_data['Spending Score (1-100)'],
                   color=cluster_colors[c], label=f'Кластер {c}', s=80, alpha=0.7)

axes[0].set_xlabel('Annual Income (k$)')
axes[0].set_ylabel('Spending Score (1-100)')
axes[0].set_title('K-Means: Income vs Spending')
axes[0].legend()
axes[0].grid(alpha=0.3)

for c in range(optimal_k):
    cluster_data = df[df['cluster'] == c]
    axes[1].scatter(cluster_data['Age'], cluster_data['Annual Income (k$)'],
                   color=cluster_colors[c], label=f'Кластер {c}', s=80, alpha=0.7)

axes[1].set_xlabel('Age')
axes[1].set_ylabel('Annual Income (k$)')
axes[1].set_title('K-Means: Age vs Income')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Бизнес-интерпретация кластеров

In [ ]:
# === Бизнес-интерпретация ===
interpretations = {
    0: 'Стандартные клиенты (средний доход, средние траты)',
    1: 'Бережливые состоятельные (высокий доход, низкие траты)',
    2: 'Молодые энтузиасты (низкий доход, высокие траты)',
    3: 'Состоятельные шопоголики (высокий доход, высокие траты) ⭐',
    4: 'Экономные (низкий доход, низкие траты)',
}

print("Бизнес-сегменты клиентов:")
print("=" * 60)
for c in range(optimal_k):
    cluster_data = df[df['cluster'] == c]
    avg_age = cluster_data['Age'].mean()
    avg_income = cluster_data['Annual Income (k$)'].mean()
    avg_spending = cluster_data['Spending Score (1-100)'].mean()
    n = len(cluster_data)
    pct = n / len(df) * 100
    interp = interpretations.get(c, f'Кластер {c}')
    print(f"\nКластер {c}: {interp}")
    print(f"  Возраст: {avg_age:.0f} | Доход: {avg_income:.0f}k$ | Spending: {avg_spending:.0f}")
    print(f"  Размер: {n} клиентов ({pct:.1f}%)")

print("\n" + "=" * 60)
print("⭐ = Приоритетный сегмент для маркетинга")

## 4. DBSCAN — поиск выбросов

In [ ]:
# DBSCAN с разными eps
for eps in [0.3, 0.5, 0.7, 1.0]:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(X_scaled)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    print(f"eps={eps}: кластеров={n_clusters}, шум={n_noise} ({n_noise/len(df)*100:.1f}%)")

In [ ]:
# Финальный DBSCAN
db = DBSCAN(eps=0.5, min_samples=5)
df['dbscan_cluster'] = db.fit_predict(X_scaled)

outliers = df[df['dbscan_cluster'] == -1]
print(f"Найдено выбросов: {len(outliers)}")
print("\nПрофиль выбросов:")
print(outliers[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].describe())

# Визуализация
plt.figure(figsize=(10, 6))
plt.scatter(df[df['dbscan_cluster'] != -1]['Annual Income (k$)'],
           df[df['dbscan_cluster'] != -1]['Spending Score (1-100)'],
           c=df[df['dbscan_cluster'] != -1]['dbscan_cluster'], cmap='viridis',
           s=60, alpha=0.6, label='Кластеры')
plt.scatter(outliers['Annual Income (k$)'], outliers['Spending Score (1-100)'],
           c='red', s=200, marker='X', label=f'Выбросы ({len(outliers)})', edgecolor='black')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('DBSCAN: Кластеры + Выбросы')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 5. PCA + визуализация

In [ ]:
# PCA: объяснённая дисперсия
pca = PCA(random_state=42)
pca.fit(X_scaled)

cumulative = np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(10, 5))
plt.plot(range(1, 4), cumulative, 'o-', linewidth=2, markersize=10)
plt.xlabel('Число компонент')
plt.ylabel('Накопленная объяснённая дисперсия')
plt.title('PCA: выбор числа компонент')
plt.grid(alpha=0.3)
plt.axhline(0.95, color='red', linestyle='--', label='95%')
plt.legend()
plt.show()

print(f"Объяснённая дисперсия: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}, PC3={pca.explained_variance_ratio_[2]:.2%}")

In [ ]:
# 2D визуализация кластеров
pca_2d = PCA(n_components=2, random_state=42)
X_pca = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
for c in range(optimal_k):
    mask = df['cluster'] == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
               color=cluster_colors[c], label=f'Кластер {c}', s=80, alpha=0.7, edgecolor='black')

# Центроиды
centers_pca = pca_2d.transform(km_final.cluster_centers_)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], c='black', s=300, marker='X', label='Центроиды')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('Кластеры в пространстве PCA')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Часть 2: Anomaly Detection на Credit Card Fraud

In [ ]:
df_fraud = pd.read_csv('/kaggle/input/creditcardfraud/creditcard.csv')

# Только нормальные признаки (без Time и Class)
feature_cols = [c for c in df_fraud.columns if c not in ['Class', 'Time']]
X_fraud = df_fraud[feature_cols].values
y_fraud = df_fraud['Class'].values

scaler = StandardScaler()
X_fraud_scaled = scaler.fit_transform(X_fraud)

print(f"Всего: {len(X_fraud_scaled)}, Fraud: {y_fraud.sum()}")
print(f"Будем считать без меток (unsupervised)")

In [ ]:
# === Isolation Forest ===
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.005,  # Примерная доля fraud (0.17%)
    random_state=42,
    n_jobs=-1
)
iso_labels = iso_forest.fit_predict(X_fraud_scaled)
iso_predictions = (iso_labels == -1).astype(int)

from sklearn.metrics import precision_score, recall_score, f1_score
print("Isolation Forest:")
print(f"  Precision: {precision_score(y_fraud, iso_predictions):.4f}")
print(f"  Recall:    {recall_score(y_fraud, iso_predictions):.4f}")
print(f"  F1:        {f1_score(y_fraud, iso_predictions):.4f}")

In [ ]:
# === Local Outlier Factor ===
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.005, novelty=False)
lof_labels = lof.fit_predict(X_fraud_scaled)
lof_predictions = (lof_labels == -1).astype(int)

print("Local Outlier Factor:")
print(f"  Precision: {precision_score(y_fraud, lof_predictions):.4f}")
print(f"  Recall:    {recall_score(y_fraud, lof_predictions):.4f}")
print(f"  F1:        {f1_score(y_fraud, lof_predictions):.4f}")

In [ ]:
# === Ансамбль методов ===
from sklearn.preprocessing import MinMaxScaler

iso_scores = -iso_forest.score_samples(X_fraud_scaled)
lof_scores = -lof.negative_outlier_factor_

scores = np.column_stack([iso_scores, lof_scores])
scores_norm = MinMaxScaler().fit_transform(scores)
ensemble_score = scores_norm.mean(axis=1)

threshold = np.percentile(ensemble_score, 99.5)
ensemble_predictions = (ensemble_score >= threshold).astype(int)

print("Ансамбль (IsoForest + LOF):")
print(f"  Precision: {precision_score(y_fraud, ensemble_predictions):.4f}")
print(f"  Recall:    {recall_score(y_fraud, ensemble_predictions):.4f}")
print(f"  F1:        {f1_score(y_fraud, ensemble_predictions):.4f}")

In [ ]:
# === Сравнение с supervised baseline ===
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_tr, X_te, y_tr, y_te = train_test_split(X_fraud_scaled, y_fraud, test_size=0.25, random_state=42, stratify=y_fraud)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_tr, y_tr)
gb_proba = gb.predict_proba(X_te)[:, 1]
gb_pred = (gb_proba >= 0.5).astype(int)

print("Gradient Boosting (supervised baseline):")
print(f"  ROC-AUC:   {roc_auc_score(y_te, gb_proba):.4f}")
print(f"  Precision: {precision_score(y_te, gb_pred):.4f}")
print(f"  Recall:    {recall_score(y_te, gb_pred):.4f}")
print(f"  F1:        {f1_score(y_te, gb_pred):.4f}")

## 📋 Выводы

1. **Сегментация:** K-Means выявил 5 чётких сегментов клиентов с разной маркетинговой стратегией
2. **Silhouette Score** помог выбрать оптимальное K
3. **DBSCAN** обнаружил аномальных клиентов (потенциально интересные случаи)
4. **PCA** снизил размерность с сохранением 95%+ дисперсии
5. **Anomaly Detection:** Isolation Forest и LOF работают без меток, но проигрывают supervised моделям